# Assignment 1: Optimizing the Delivery Promise for Rosa's Pizza
**Course**: BUSADMIN O712

In [2]:
# !pip install -q git+https://github.com/zhouy185/rosa-starter.git

import numpy as np
import pandas as pd
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

print("Starter package initialized")
print(f"Operational Zones ({len(ZONES)}): {ZONES}")
print(f"Operational Time Blocks ({len(TIME_BLOCKS)}): {TIME_BLOCKS}")
print(f"Cost Structure Parameters: {COSTS}")
print(f"Baseline Delivery Promise: {PROMISE} minutes")

Starter package initialized
Operational Zones (3): ['Central', 'North', 'Far West']
Operational Time Blocks (4): ['Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other']
Cost Structure Parameters: {'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}
Baseline Delivery Promise: 45 minutes


---

## Part I. Quantitative Diagnosis of Current Operations

### Part I (a): Late Order Rate Function Formulation & Verification

To evaluate delivery reliability across variable operational granularities, we define `calculate_late_percentage`. The function accepts individual zone and time-block specifications, or the universal `'all'` designator to compute aggregated delay percentages across multi-cell partitions.

For aggregate scopes, the calculation aggregates raw order counts and overdue instances across all relevant cells, thereby avoiding distortion from unweighted cell averaging.


In [3]:
def calculate_late_percentage(zone, time_block, promise=PROMISE, seed=42):
    """
    Calculate the percentage of late deliveries for specified zone and time block scopes.
    
    Parameters
    ----------
    zone : str
        Target geographic zone ('Central', 'North', 'Far West', or 'all').
    time_block : str
        Target operational window ('Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other', or 'all').
    promise : float or int, optional
        Promised delivery time threshold in minutes (default is 45).
    seed : int, optional
        Random seed for simulator reproducibility (default is 42).
        
    Returns
    -------
    float
        Percentage of late orders rounded to two decimal places.
    """
    target_zones = ZONES if zone == "all" else [zone]
    target_blocks = TIME_BLOCKS if time_block == "all" else [time_block]
    
    total_orders = 0
    total_late = 0
    
    for z in target_zones:
        for tb in target_blocks:
            times = delivery_times(z, tb, promise, seed=seed)
            total_orders += len(times)
            total_late += int(np.sum(times > promise))
            
    if total_orders == 0:
        return 0.0
        
    late_percentage = (total_late / total_orders) * 100.0
    return round(late_percentage, 2)


# Test cases covering all four scope configurations under the 45-minute baseline promise
test_cases_late = [
    ("Specific Zone & Specific Time Block", "Central", "Weekday eve"),
    ("Specific Zone Across All Time Blocks", "Far West", "all"),
    ("All Zones for a Specific Time Block", "all", "Lunch"),
    ("Global Scope Across All Zones & Blocks", "all", "all")
]

results_late_a = []
for description, zone_arg, time_arg in test_cases_late:
    rate = calculate_late_percentage(zone_arg, time_arg, promise=45, seed=42)
    results_late_a.append({
        "Scope Type": description,
        "Zone Argument": zone_arg,
        "Time Block Argument": time_arg,
        "Late Order Rate (%)": f"{rate:.2f}%"
    })

pd.DataFrame(results_late_a)


,Scope Type,Zone Argument,Time Block Argument,Late Order Rate (%)
0,Specific Zone & Specific Time Block,Central,Weekday eve,6.97%
1,Specific Zone Across All Time Blocks,Far West,all,15.90%
2,All Zones for a Specific Time Block,all,Lunch,3.68%
3,Global Scope Across All Zones & Blocks,all,all,9.29%


### Part I (b): Operational Cell Ranking by Late Order Rate


In [ ]:
# Rank all 12 operational permutations by late arrival rate
ranking_data_b = []

for z in ZONES:
    for tb in TIME_BLOCKS:
        times = delivery_times(z, tb, promise=PROMISE, seed=42)
        n_orders = len(times)
        n_late = int(np.sum(times > PROMISE))
        rate = (n_late / n_orders) * 100.0 if n_orders > 0 else 0.0
        ranking_data_b.append({
            "Zone": z,
            "Time Block": tb,
            "Total Orders": n_orders,
            "Late Orders": n_late,
            "Late Order Rate (%)": round(rate, 2)
        })

df_rank_late = pd.DataFrame(ranking_data_b).sort_values(by="Late Order Rate (%)", ascending=False).reset_index(drop=True)

df_rank_late.index += 1

df_rank_late


,Zone,Time Block,Total Orders,Late Orders,Late Order Rate (%)
1,Far West,Fri/Sat eve,210,77,36.67
2,North,Fri/Sat eve,150,27,18.00
3,Far West,Weekday eve,170,18,10.59
4,North,Weekday eve,250,23,9.20
5,Central,Fri/Sat eve,190,15,7.89
6,Central,Weekday eve,330,23,6.97
7,Far West,Other,192,12,6.25
8,North,Lunch,190,8,4.21
9,Central,Lunch,260,10,3.85
10,North,Other,160,4,2.50


### Part I (c): Average Delivery Time Function Formulation & Verification


In [5]:
def calculate_average_delivery_time(zone, time_block, promise=PROMISE, seed=42):
    """
    Calculate the pooled average delivery time for specified zone and time block scopes.
    
    Parameters
    ----------
    zone : str
        Target geographic zone ('Central', 'North', 'Far West', or 'all').
    time_block : str
        Target operational window ('Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other', or 'all').
    promise : float or int, optional
        Promised delivery time threshold in minutes (default is 45).
    seed : int, optional
        Random seed for simulator reproducibility (default is 42).
        
    Returns
    -------
    float
        Average delivery duration in minutes rounded to two decimal places.
    """
    target_zones = ZONES if zone == "all" else [zone]
    target_blocks = TIME_BLOCKS if time_block == "all" else [time_block]
    
    delivery_records = []
    for z in target_zones:
        for tb in target_blocks:
            times = delivery_times(z, tb, promise, seed=seed)
            if len(times) > 0:
                delivery_records.append(times)
                
    if not delivery_records:
        return 0.0
        
    pooled_times = np.concatenate(delivery_records)
    return round(float(np.mean(pooled_times)), 2)


# Test cases covering all four scope configurations under the 45-minute baseline promise
test_cases_time = [
    ("Specific Zone & Specific Time Block", "Central", "Weekday eve"),
    ("Specific Zone Across All Time Blocks", "Far West", "all"),
    ("All Zones for a Specific Time Block", "all", "Lunch"),
    ("Global Scope Across All Zones & Blocks", "all", "all")
]

results_time_c = []
for description, z_arg, tb_arg in test_cases_time:
    avg_duration = calculate_average_delivery_time(z_arg, tb_arg, promise=45, seed=42)
    results_time_c.append({
        "Scope Type": description,
        "Zone Argument": z_arg,
        "Time Block Argument": tb_arg,
        "Average Delivery Time (min)": f"{avg_duration:.2f}"
    })

pd.DataFrame(results_time_c)


,Scope Type,Zone Argument,Time Block Argument,Average Delivery Time (min)
0,Specific Zone & Specific Time Block,Central,Weekday eve,29.98
1,Specific Zone Across All Time Blocks,Far West,all,35.38
2,All Zones for a Specific Time Block,all,Lunch,28.49
3,Global Scope Across All Zones & Blocks,all,all,31.56


### Part I (d): Operational Cell Ranking by Mean Delivery Duration


In [6]:
# Rank all 12 operational permutations by mean delivery duration
ranking_data_d = []

for z in ZONES:
    for tb in TIME_BLOCKS:
        avg_time = calculate_average_delivery_time(z, tb, promise=PROMISE, seed=42)
        ranking_data_d.append({
            "Zone": z,
            "Time Block": tb,
            "Average Delivery Time (min)": avg_time
        })

df_rank_time = pd.DataFrame(ranking_data_d).sort_values(
    by="Average Delivery Time (min)", ascending=False
).reset_index(drop=True)

df_rank_time.index += 1
df_rank_time


,Zone,Time Block,Average Delivery Time (min)
1,Far West,Fri/Sat eve,42.98
2,North,Fri/Sat eve,37.07
3,Far West,Weekday eve,35.25
4,Central,Fri/Sat eve,32.39
5,North,Weekday eve,31.94
6,Far West,Lunch,30.37
7,Far West,Other,30.33
8,Central,Weekday eve,29.98
9,North,Lunch,28.91
10,North,Other,27.94


### Part I (e): Comparative Mechanism Analysis: Late Rate versus Mean Delivery Time

To determine whether the late order rate or the mean delivery duration provides more significant operational relevance for Rosa's decision-making, we analyze the governing cost mechanics, statistical distributions, and empirical rank discordance between the two metrics.

Based on empirical evidence and financial structure, **the late order rate is significantly more relevant to Rosa's operational and strategic decisions** due to two underlying reasons:

#### 1. Direct Structural Coupling to Loss
Rosa's cost structure imposes penalties strictly upon individual threshold breach events. Specifically, each delivery completed past the promised time triggers an immediate cash refund of \$10.00 and an expected attrition loss of 1.8 future customer orders (representing \$16.20 in foregone profit margin at \$9.00 per order). This produces a step-penalty of \$26.20 per late transaction (Oh, Part 2 has covered this!). Mean delivery duration does not enter the income statement directly; only the discrete count of orders in the tail exceeding the promised threshold drives financial losses.

#### 2. Empirical Rank Discordance Across Operational Cells
Comparing the empirical rankings from Sections 1.2 and 1.4 reveals critical discrepancies where mean delivery duration fails to predict operational risk:
- **`Far West, Lunch` vs. `Central, Weekday eve`**: In the mean delivery duration ranking (Section 1.4), `Far West, Lunch` ranks 6th with an average time of 30.37 minutes, appearing slower than `Central, Weekday eve` (ranked 8th at 29.98 minutes). However, in the late rate ranking (Section 1.2), `Central, Weekday eve` incurs a late rate of 6.97% (ranked 6th), whereas `Far West, Lunch` incurs a late rate of only 2.50% (ranked 11th). Focusing on average delivery duration would direct managerial intervention toward `Far West, Lunch`, whereas `Central, Weekday eve` generates nearly three times the rate of costly service failures.
- **`Far West, Other` vs. `North, Weekday eve`**: `Far West, Other` exhibits an average duration of 30.33 minutes, which is faster than `North, Weekday eve` at 31.94 minutes. Yet `Far West, Other` generates a lower late rate (6.25%) compared to `North, Weekday eve` (9.20%), reflecting localized tail compression.


---

## Part II. Mathematical Optimization of Delivery Promises


### Part II (a): Unit Cost Calculation per Delinquent Order


In [7]:
def calculate_late_cost_per_order(costs=COSTS):
    """
    Compute the total financial penalty incurred by a single late delivery order.
    
    Parameters
    ----------
    costs : dict
        Cost dictionary containing 'refund', 'churn_orders', and 'margin'.
        
    Returns
    -------
    float
        Total cost penalty per late delivery.
    """
    direct_refund = costs["refund"]
    churn_loss = costs["churn_orders"] * costs["margin"]
    total_penalty = direct_refund + churn_loss
    return round(float(total_penalty), 2)


# Compute and verify the unit late penalty using baseline cost specifications
unit_penalty = calculate_late_cost_per_order(COSTS)
print(f"Direct Refund per Late Order:       ${COSTS['refund']:.2f}")
print(f"Churned Future Orders:              {COSTS['churn_orders']:.1f} orders")
print(f"Profit Margin per Order:            ${COSTS['margin']:.2f}")
print(f"Implied Churn Loss per Late Order:  ${COSTS['churn_orders'] * COSTS['margin']:.2f}")
print(f"--------------------------------------------------")
print(f"Total Financial Penalty per Late Order: ${unit_penalty:.2f}")


Direct Refund per Late Order:       $10.00
Churned Future Orders:              1.8 orders
Profit Margin per Order:            $9.00
Implied Churn Loss per Late Order:  $16.20
--------------------------------------------------
Total Financial Penalty per Late Order: $26.20


### Part II (b): Search Space Delineation and Optimization Engine Formulation

#### Discussion on Candidate Promise Range Selection
To establish an effective discrete parameter grid for evaluating alternative delivery promises, we define the search bounds and step size based on operational feasibility:

1. **Step Size Justification (5-Minute Granularity)**: Customers process delivery commitments through discrete intervals (e.g., 30, 35, 40, 45, 50 minutes). Operational promises with arbitrary non-integer precision (e.g., 37 or 43 minutes) create consumer confusion. A 5-minute step size maintains high resolution across the decision surface while preserving computational tractability.
2. **Lower Bound Feasibility (20 Minutes)**: Food preparation and delivery impose a strict physical floor on delivery times. Setting a promise threshold below 20 minutes triggers near-total delinquency (even Domino's can't do this which is the fastest in my mind). The resultant penalty liabilities of \$26.20 per order rapidly outstrip the \$9.00 unit margin, producing catastrophic operational losses.
3. **Upper Bound Feasibility (60 Minutes)**: Market demand follows a logistic decay function relative to delivery promises. Beyond 60 minutes, customer conversion drops toward zero as consumers substitute alternatives. Total gross margin collapses, rendering long promises economically unviable despite zero delinquency.

Therefore, evaluating candidate promises across the interval of **[20, 60] minutes in 5-minute increments** captures the global profit peak across all operating blocks.


In [8]:
def choose_best_promise(zone, time_block, promises, costs=COSTS, seed=42):
    """
    Identify the optimal promised delivery time that maximizes net operational profit.
    
    Parameters
    ----------
    zone : str
        Target geographic zone ('Central', 'North', or 'Far West').
    time_block : str
        Target operational window ('Lunch', 'Weekday eve', 'Fri/Sat eve', or 'Other').
    promises : list or iterable of int/float
        Candidate delivery promise values (in minutes) to evaluate.
    costs : dict, optional
        Cost parameters specifying refund, churn_orders, and margin (default is COSTS).
    seed : int, optional
        Random seed for simulator reproducibility (default is 42).
        
    Returns
    -------
    dict
        Dictionary containing optimal promise, peak net profit, and full evaluation trajectory.
    """
    penalty_per_late = calculate_late_cost_per_order(costs)
    margin = costs["margin"]
    
    best_promise = None
    best_net_profit = float("-inf")
    evaluation_records = []
    
    for p in promises:
        times = delivery_times(zone, time_block, promise=p, seed=seed)
        n_orders = len(times)
        
        if n_orders == 0:
            n_late = 0
            late_rate = 0.0
            gross_margin = 0.0
            late_costs = 0.0
            net_profit = 0.0
        else:
            n_late = int(np.sum(times > p))
            late_rate = (n_late / n_orders) * 100.0
            gross_margin = n_orders * margin
            late_costs = n_late * penalty_per_late
            net_profit = gross_margin - late_costs
            
        evaluation_records.append({
            "Promise (min)": p,
            "Total Orders": n_orders,
            "Late Orders": n_late,
            "Late Rate (%)": round(late_rate, 2),
            "Gross Margin ($)": round(gross_margin, 2),
            "Late Costs ($)": round(late_costs, 2),
            "Net Profit ($)": round(net_profit, 2)
        })
        
        if net_profit > best_net_profit:
            best_net_profit = net_profit
            best_promise = p
            
    df_eval = pd.DataFrame(evaluation_records)
    
    return {
        "best_promise": best_promise,
        "best_net_profit": round(best_net_profit, 2),
        "evaluation_table": df_eval
    }


# Execute optimization benchmark on the severe bottleneck cell: Far West, Fri/Sat eve
candidate_promises = list(range(20, 65, 5))
benchmark_result = choose_best_promise(
    zone="Far West",
    time_block="Fri/Sat eve",
    promises=candidate_promises,
    costs=COSTS,
    seed=42
)

print(f"Optimal Promise for Far West (Fri/Sat eve): {benchmark_result['best_promise']} minutes")
print(f"Maximized Net Profit: ${benchmark_result['best_net_profit']:.2f}")
benchmark_result["evaluation_table"]


Optimal Promise for Far West (Fri/Sat eve): 55 minutes
Maximized Net Profit: $1186.20


,Promise (min),Total Orders,Late Orders,Late Rate (%),Gross Margin ($),Late Costs ($),Net Profit ($)
0,20,241,241,100.00,2169.0,6314.2,-4145.2
1,25,239,235,98.33,2151.0,6157.0,-4006.0
2,30,237,219,92.41,2133.0,5737.8,-3604.8
3,35,232,178,76.72,2088.0,4663.6,-2575.6
4,40,224,130,58.04,2016.0,3406.0,-1390.0
5,45,210,77,36.67,1890.0,2017.4,-127.4
6,50,188,35,18.62,1692.0,917.0,775.0
7,55,158,9,5.70,1422.0,235.8,1186.2
8,60,121,1,0.83,1089.0,26.2,1062.8


### Empirical Trajectory and Optimization Insights for Far West (Fri/Sat eve)

The grid evaluation on the bottleneck cell (`Far West, Fri/Sat eve`) demonstrates the interplay between demand volume and fulfillment penalties:

1. **Under-Promising Regime (20 to 40 minutes)**: Although order volumes remain high (224 to 241 orders), the physical system cannot satisfy these commitments, producing late rates between 58.0% and 100.0%. At a 20-minute promise, cumulative late penalties reach \$6,314.20, generating a net operating deficit of **-\$4,145.20**.
2. **Baseline Operations (45 minutes)**: Under Rosa's current 45-minute promise, the cell generates 210 orders with 77 late deliveries (36.67% late rate). The resulting late penalties of \$2,017.40 exceed total gross margin (\$1,890.00), yielding a net loss of **-\$127.40**. Rosa currently sustains an operating deficit on every weekend evening in the Far West zone.
3. **Global Optimum (55 minutes)**: Increasing the promised duration to 55 minutes causes a controlled demand contraction from 210 to 158 orders (reducing gross margin to \$1,422.00). However, the expanded time window relieves dispatch pressure, dropping late deliveries from 77 to 9 (a late rate reduction from 36.67% to 5.70%). Delinquency penalties collapse to \$235.80, elevating net profit to **+\$1,186.20**.
4. **Over-Conservative (60 minutes)**: Further extending the promise to 60 minutes drives late deliveries down to 1 order (0.83% late rate). However, this results in a significant drop in total orders to 121, pulling net profit down to \$1,062.80.

Consequently, shifting from 45 minutes to 55 minutes produces an immediate profit turnaround of **+\$1,313.60** over the four-week period in this single operational block.


### Part III. Streamlit Web App

Please access: https://rosas-pizza-delivery-optimizer-uhfma4chvpymsmskqj6bay.streamlit.app

---

## Appendix: AI Assistance and Prompt Engineering Log

AI usages were mainly concentrated in the following areas:

### 1. Architectural Roles and Scope of AI Interaction
1. **Architectural Design**: Facilitating the overall structure and components of the delivery optimization system, including data flow, user interface, and backend logic.
2. **Implementation Support**: Providing code snippets, algorithmic logic, and debugging assistance for the optimization engine, cost accounting functions, and Streamlit application. Human code review and AI code review were in the loop.
3. **Writing Support**: Assisting in drafting documentation and explanatory narratives.

### 2. User Directives and Prompt Engineering Record (Only the most significant prompts are listed)

| No. | Prompt | Operational Objective & Impact |
| :--- | :--- | :--- |
| **P1** | *Read the assignment requirements, list all objectives and constraints. Ask me 10 questions to clarify any ambiguities.* | Parsed the PDF assignment specification, extracted requirements, identified the fundamental demand-fulfillment trade-off, and outlined the development roadmap across Part I, Part II, and Part III. This step is to make sure AI is consistent with human intent. |
| **P2** | *I have set up the venv with uv. Create a Jupyter notebook for Part I and Part II. Import the provided values and delivery_times from starter; do not recreate or modify the starter code. Write functions for late-order percentage and average delivery time that support a specific zone/time block and the four combinations involving "all". Write the essential test cases, but leave the test arguments unspecified and I will fill them in later. Rank all 12 zone–time-block pairs by late rate and by average delivery time. For documentation or any text content, follow the linguistic and structural style defined in ~/Documents/my-voice-dna-technical.md.* | Initialized an isolated Python virtual environment via `uv`, constructed the full mathematical implementations for Part I and Part II, and authored all code and narrative text in rigorous technical English following Voice DNA guidelines. |
| **P3** | *Use /handoff to wrap up the current tasks.* | Prepare to hand over the completed tasks to the next phase of development. Clear the context window.|
| **P4** | *Read handover.md, continue completing the remaining tasks. Use /notebook-to-streamlit skill to create the Streamlit web application.* | Continue the development of the Streamlit web application, ensuring all features are implemented and tested. Construct the production Streamlit web application (`app.py`), configured deployment specifications (`requirements.txt`), published the remote GitHub repository, and integrated active production URLs into the notebook. |
| **P5** | Update the technical documentation in README.md and cross-validate all implementation details. Raise all concerns and document any issues found. Do not fix any issues at this time, discuss with me first. | This is performed in Codex (6 Sol Medium) while all steps above are performed in Antigravity (3.8 Flash Medium)
